# Does σ track the true uncertainty? — a per-region, per-variable study

**Standalone uncertainty chapter.** The calibration work in `analyze_cross_lead.ipynb`
asked whether the predictive is calibrated *on average and across lead*. Here we ask the
sharper, lead-independent question: **does the predicted spread σ actually track the true
predictive uncertainty, case by case?** — broken down **per region** and **per variable**,
on the main single-region snapshot experiments (no forecast lead).

## The epistemic problem: we never observe the "true uncertainty"

For a single station-time we see **one** realization `y` and a predicted distribution
`(μ, σ)`. The "true uncertainty" — the conditional spread of the error given everything the
model knows — is not directly observable. We make it checkable by **conditioning and
pooling**: for *any* grouping of observations that share a property `z`, the realized
`RMSE(z) = √(E[(y−μ)² | z])` is an unbiased estimate of the average error magnitude for
cases with that property. Grouping by the predicted σ itself gives the identity a good
uncertainty must satisfy,

$$\mathbb{E}\big[(y-\mu)^2 \,\big|\, \sigma\big] = \sigma^2 ,$$

i.e. **within a set of cases the model calls equally uncertain, the realized error should
equal the σ it advertised.** "True uncertainty" for a group is simply its realized error,
revealed by pooling.

## Reliability is not enough — you also need resolution

A σ can be right *on average* yet carry *no case-level information*. We therefore judge σ on
two orthogonal axes (a reliability–resolution split, à la Murphy):

- **Reliability (calibration):** σ matches the error *on average*. Diagnosed by the
  spread–skill ratio `SSR = RMS(σ)/RMSE ≈ 1`, PIT uniformity (`CE_TV`), and central-interval
  coverage.
- **Resolution (discrimination / sharpness-that-tracks):** σ *varies across cases*, and that
  variation lines up with the actual error. Diagnosed by
  (i) the spread–skill **relationship** — bin by σ and check binned-RMSE rises 1:1 with
  binned-σ; (ii) the rank correlation `ρ(σ, |error|)`; (iii) `CV(σ)=std(σ)/mean(σ)` (does σ
  vary at all); and (iv) the **value of a varying σ** — the NLL a global rescaling of σ can
  *not* recover (`resolution gap = NLL(best constant σ) − NLL(best-scaled σ)`, in nats).

The distinction is the whole point. Our third method makes it concrete. The **ERA5→station
bilinear interpolation** is a *deterministic* point predictor — it has no uncertainty head. We
turn it into a probabilistic forecast the only way a deterministic model can: wrap it in a
homoscedastic Gaussian `N(interp, σ_c²)` whose σ is a **single global constant**, set to the
std of its own residuals (`residual_std = std(pred − target)`, in `tessera-baselines`).
So its "uncertainty" is one climatological error bar reused at every station — not a predicted,
case-varying σ. Because σ_c is fit to the residuals it is *handed* an oracle-calibrated average
spread (`SSR≈1`) yet *still* has **zero** resolution — it cannot tell a hard case from an easy
one. That is exactly why it is the null model against which "σ tracks uncertainty" is measured:
the comparison is deliberately generous on reliability so any gap is purely resolution.

## A third lens: does σ track *known physical* drivers of uncertainty?

Beyond self-consistency, we can ask whether σ is large where we *independently know*
downscaling is hard — chiefly **complex terrain** (large sub-grid elevation anomaly `|Δelev|`,
where a coarse grid cell hides large within-cell variation). If σ tracks true uncertainty it
should (a) correlate with `|Δelev|` across stations, and (b) hold **conditional coverage**
across terrain — whereas a globally-calibrated *constant* σ must over-cover flat terrain and
under-cover complex terrain.

## What we compare

| method | σ | role |
|---|---|---|
| **ERA5 interp** (`*_era5_interp_baseline`) | **constant** = std of residuals (deterministic point pred. + post-hoc σ) | homoscedastic null model — reliability without resolution |
| **ConvCNP, no TESSERA** (`*_bilinear_baseline_mtpi_wd`) | learned, heteroscedastic | the downscaler without the embedding |
| **ConvCNP + TESSERA** (`*_vae_lat16_concat_with_elev_mtpi_no_static_wd`) | learned, heteroscedastic | lat-16 direct-concat TESSERA, + elevation + mTPI |

**Matrix:** 5 regions {Europe, US, East Asia, Australia, S. Africa} × 2 variables {t2m —
Gaussian head; wind — truncated-Normal head} × 3 methods × seeds {42, 123, 456}, pooled over
seeds. Both heads are fully parametric, so the PIT, moments, and NLL are **closed form** and
evaluated exactly over every one of the ~10⁵–10⁶ test observations per cell (Gaussian CDF for
t2m; left-truncated-Normal CDF `(Φ((y−μ)/σ)−Φ(α))/(1−Φ(α))`, `α=−μ/σ`, for wind). Reads only
`test_predictions.npz` / `test_station_errors.npz` / `test_summary.json` — no torch.


In [ ]:
# === Configuration + closed-form per-observation statistics =================
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from scipy.special import log_ndtr
from scipy.stats import norm, spearmanr

%matplotlib inline
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 150,
        "font.size": 10,
        "axes.grid": True,
        "grid.alpha": 0.25,
    }
)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# Per-region training_runs_snapshot_14y_<region>/ dirs under $TESSERA_DATA_ROOT.
from tessera_downscaling.paths import training_runs_dir


def root(region):
    return training_runs_dir(f"snapshot_14y_{region}")


OUT_DIR = Path("uncertainty_analysis_outputs")
OUT_DIR.mkdir(exist_ok=True)

REGIONS = ["eu", "us", "east_asia", "australia", "southern_africa"]
REGION_LABEL = {
    "eu": "Europe",
    "us": "US",
    "east_asia": "East Asia",
    "australia": "Australia",
    "southern_africa": "S. Africa",
}
SEEDS = [42, 123, 456]
VARS = {"t2m": "gaussian", "wind": "truncated_normal"}  # variable -> head

# Method -> config stem. era5_interp is the constant-σ homoscedastic null model.
METHODS = ["era5_interp", "baseline", "tessera"]
METHOD_LABEL = {
    "era5_interp": "ERA5 interp (const σ)",
    "baseline": "ConvCNP (no TESSERA)",
    "tessera": "ConvCNP + TESSERA",
}
METHOD_COLOR = {"era5_interp": "#b0b0b0", "baseline": "#c1440e", "tessera": "#1f77b4"}


def stem(var, method):
    if method == "era5_interp":
        return f"{var}_snap_era5_interp_baseline"
    tail = (
        "bilinear_baseline_mtpi_wd"
        if method == "baseline"
        else "vae_lat16_concat_with_elev_mtpi_no_static_wd"
    )
    # config token order differs: `wind_truncnormal_snap_…` vs `t2m_snap_…`
    return f"wind_truncnormal_snap_{tail}" if var == "wind" else f"t2m_snap_{tail}"


_LOG2PI = float(np.log(2 * np.pi))
C1, C2 = (
    0.3413447461,
    0.4772498681,
)  # half-widths of the 68.27% / 95.45% central intervals
NB_PIT = 20


# ---- closed-form predictive (mean, std, PIT) and NLL, per head -------------
def _gauss(mu, sigma, y):
    return mu, sigma, norm.cdf((y - mu) / sigma)


def _tnorm(mu, sigma, y):  # left-truncated Normal on [0, ∞)
    # Log-domain (log_ndtr) forms — STABLE in the deeply-truncated calm-wind
    # regime μ/σ ≪ 0, where the naive 1−Φ(−μ/σ) underflows and would collapse the
    # predictive mean onto μ (large negative). Matches evaluate.py / heads.py.
    s = mu / sigma  # = −α
    logZ = log_ndtr(s)  # log P(X>0) = log(1−Φ(α))
    lam = np.exp((-0.5 * s * s - 0.5 * _LOG2PI) - logZ)  # φ(α)/Z, stable (→|s| as s→−∞)
    mean = mu + sigma * lam
    var = sigma**2 * np.clip(1.0 - s * lam - lam**2, 0.0, None)  # σ²(1+αλ−λ²)
    pit = np.where(
        y >= 0.0, np.clip(-np.expm1(log_ndtr((mu - y) / sigma) - logZ), 0.0, 1.0), 0.0
    )
    return mean, np.sqrt(var), pit


def _nll_gauss(mu, sigma, y):
    return 0.5 * (_LOG2PI + 2 * np.log(sigma) + ((y - mu) / sigma) ** 2)


def _nll_tnorm(mu, sigma, y):
    ys = np.clip(y, 1e-5, None)
    z = (ys - mu) / sigma
    return -((-0.5 * z * z - 0.5 * _LOG2PI) - np.log(sigma) - log_ndtr(mu / sigma))


def _nll_fn(head):
    return _nll_gauss if head in ("gaussian", "const") else _nll_tnorm


def _min_1d(f, grid):
    best = (np.inf, grid[0])
    for c in grid:
        v = f(c)
        if v < best[0]:
            best = (v, c)
    for c in np.linspace(best[1] * 0.6, best[1] * 1.5, 40):  # local refine
        v = f(c)
        if v < best[0]:
            best = (v, c)
    return best


def best_const_sigma_nll(head, mu, y):
    """min over a CONSTANT σ of mean NLL (holding μ fixed) — no resolution possible."""
    nf = _nll_fn(head)
    if head in ("gaussian", "const"):
        mse = np.mean((y - mu) ** 2)
        return 0.5 * (_LOG2PI + np.log(mse) + 1.0)
    return _min_1d(
        lambda c: nf(mu, np.full_like(mu, c), y).mean(), np.geomspace(0.05, 20, 60)
    )[0]


def best_scale_nll(head, mu, sigma, y):
    """min over a global SCALE of σ (σ→c·σ) — keeps resolution, fixes spread scale.
    Closed form for the Gaussian (c*=RMS(err/σ)); 1-D search for the truncated Normal."""
    if head in ("gaussian", "const"):
        z2 = np.mean(((y - mu) / sigma) ** 2)  # optimal c² = mean((err/σ)²)
        return 0.5 * (_LOG2PI + 2 * np.mean(np.log(sigma)) + np.log(z2) + 1.0)
    return _min_1d(
        lambda c: _nll_tnorm(mu, c * sigma, y).mean(), np.geomspace(0.3, 3.0, 40)
    )[0]


print(
    "Config ready. Regions:",
    REGIONS,
    "| roots exist:",
    {r: root(r).exists() for r in REGIONS},
)

In [ ]:
# === Loaders + reliability/resolution metrics + full pass ===================
def load_cell(region, var, method, seed):
    """Per-observation (mean, std, σ, PIT, y, NLL, station idx) for one run."""
    f = root(region) / f"{stem(var, method)}_seed{seed}" / "test_predictions.npz"
    if not f.exists():
        return None
    with np.load(f, allow_pickle=True) as z:
        if f"{var}_param_mu" in z:  # learned heteroscedastic head
            mu = z[f"{var}_param_mu"].astype(np.float64)
            sigma = np.exp(
                0.5 * np.clip(z[f"{var}_param_log_var"].astype(np.float64), -10, 10)
            )
            y = z[f"{var}_targets"].astype(np.float64)
            sidx = z[f"{var}_station_indices"].astype(np.int64)
            head = VARS[var]
            mean, std, pit = (_gauss if head == "gaussian" else _tnorm)(mu, sigma, y)
            nll = (_nll_gauss if head == "gaussian" else _nll_tnorm)(mu, sigma, y)
        else:  # ERA5 interp: constant σ
            mu = z[f"{var}_predictions"].astype(np.float64)
            y = z[f"{var}_targets"].astype(np.float64)
            sigma = z[f"{var}_predicted_stds"].astype(np.float64)
            sidx, head = None, "const"
            mean, std, pit = _gauss(mu, sigma, y)  # naive Gaussian reference
            nll = _nll_gauss(mu, sigma, y)
    ok = np.isfinite(mean) & np.isfinite(std) & np.isfinite(pit) & np.isfinite(y)
    sig = sigma[ok] if np.ndim(sigma) else np.full(int(ok.sum()), float(sigma))
    return dict(
        mean=mean[ok],
        std=std[ok],
        sigma=sig,
        pit=pit[ok],
        y=y[ok],
        nll=nll[ok],
        mu=mu[ok],
        head=head,
        sidx=(sidx[ok] if sidx is not None else None),
    )


def load_station_err(region, var, method, seed):
    f = root(region) / f"{stem(var, method)}_seed{seed}" / "test_station_errors.npz"
    if not f.exists():
        return None
    with np.load(f, allow_pickle=True) as z:
        return {k: z[k] for k in z.keys()}


def _reliability(std, pit, err):
    h = np.histogram(pit, bins=NB_PIT, range=(0, 1))[0]
    return dict(
        ssr=np.sqrt(np.mean(std**2)) / np.sqrt(np.mean(err**2)),
        cov68=100 * np.mean(np.abs(pit - 0.5) < C1),
        cov95=100 * np.mean(np.abs(pit - 0.5) < C2),
        ce_tv=0.5 * np.abs(h / h.sum() - 1 / NB_PIT).sum(),
        hist=h,
    )


def _resolution(sigma, err, nll, head, mu, y, nq=10, sub=1_000_000):
    cv = np.std(sigma) / np.mean(sigma)
    n = len(sigma)
    idx = (
        np.random.default_rng(0).choice(n, sub, replace=False)
        if n > sub
        else slice(None)
    )
    rho = spearmanr(sigma[idx], np.abs(err[idx])).correlation if cv > 1e-9 else 0.0
    bx, by = [], []
    if cv > 1e-9:  # spread-skill relationship (σ deciles)
        qs = np.quantile(sigma, np.linspace(0, 1, nq + 1))
        qs[-1] += 1e-9
        b = np.clip(np.digitize(sigma, qs[1:-1]), 0, nq - 1)
        for k in range(nq):
            s = b == k
            if s.sum() >= 50:
                bx.append(np.sqrt(np.mean(sigma[s] ** 2)))
                by.append(np.sqrt(np.mean(err[s] ** 2)))
    nll_const = best_const_sigma_nll(head, mu, y)  # NLL decomposition (nats)
    nll_recal = best_scale_nll(head, mu, sigma, y)
    nll_model = float(np.mean(nll))
    return dict(
        cv_sigma=cv,
        spearman_sig_err=float(rho),
        ss_x=np.array(bx),
        ss_y=np.array(by),
        nll_model=nll_model,
        resolution_gap=float(nll_const - nll_recal),
        calib_gap=float(nll_model - nll_recal),
    )


def process_all(regions=REGIONS, variables=list(VARS), methods=METHODS, seeds=SEEDS):
    rows, ss_curves, pit_hists = [], {}, {}
    for region in regions:
        for var in variables:
            for method in methods:
                for seed in seeds:
                    c = load_cell(region, var, method, seed)
                    if c is None:
                        continue
                    err = c["mean"] - c["y"]
                    rel = _reliability(c["std"], c["pit"], err)
                    res = _resolution(
                        c["sigma"], err, c["nll"], c["head"], c["mu"], c["y"]
                    )
                    rows.append(
                        dict(
                            region=region,
                            var=var,
                            method=method,
                            seed=seed,
                            n=len(c["y"]),
                            rmse=np.sqrt(np.mean(err**2)),
                            mean_sigma=np.mean(c["sigma"]),
                            **{k: rel[k] for k in ["ssr", "cov68", "cov95", "ce_tv"]},
                            **{
                                k: res[k]
                                for k in [
                                    "cv_sigma",
                                    "spearman_sig_err",
                                    "nll_model",
                                    "resolution_gap",
                                    "calib_gap",
                                ]
                            },
                        )
                    )
                    pit_hists[(region, var, method)] = (
                        pit_hists.get((region, var, method), 0) + rel["hist"]
                    )
                    ss_curves.setdefault((region, var, method), []).append(
                        (res["ss_x"], res["ss_y"])
                    )
    return pd.DataFrame(rows), ss_curves, pit_hists


print("Running full pass (5 regions × 2 vars × 3 methods × 3 seeds)… ~40s")
unc_df, SS, PH = process_all()
unc_df["method"] = pd.Categorical(unc_df["method"], categories=METHODS, ordered=True)
print(f"Done: {len(unc_df)} rows.")

## 1. Reliability — is σ right *on average*?

The necessary-but-not-sufficient first test. `SSR = RMS(σ)/RMSE` (ideal 1), central-68.3%
coverage (PIT-based, exact for both heads), and the PIT calibration error
`CE_TV = ½·Σ|p_b − 1/B|` (0 = uniform PIT). Watch the constant-σ ERA5 baseline: it can reach
`SSR ≈ 1` (its σ is tuned to the overall RMSE) yet its PIT is far from uniform — reliability
of the *second moment* does not make the *shape* right, and says nothing about resolution.


**Why the PIT is not redundant with SSR / coverage.** Each predicted distribution *is*
fixed by its two parameters — but that pins down what the model *claims*, not whether the
*observations* obey it. SSR compares one aggregate second moment; k-σ coverage checks one or
two probability levels. The PIT `z=F(y)` checks agreement at **every** quantile at once, per
observation. For a Gaussian a uniform PIT *implies* `SSR=1` and correct coverage at all levels,
but **not** conversely: the constant-σ ERA5 model scores `SSR=1.000` yet `cov68≈78%` (not 68)
and `CE_TV≈0.15` — its residuals are heavier-tailed than any Gaussian, a 4th-moment mismatch a
2nd-moment statistic cannot see. So the PIT adds the shape the moments miss (tails/kurtosis,
skew) and catches heteroscedastic miscalibration that cancels in the average. For the
truncated-Normal it is the *only* exact reliability check (the σ-interval coverage is a proxy,
and the calm-wind atom at 0 shows up only as a PIT spike). None of these, PIT included, measure
resolution — that is §2.

In [ ]:
# === Reliability: tables + PIT rank histograms (pooled over regions) ========
def _pivot(df, var, col, fmt):
    a = (
        df[df["var"] == var]
        .groupby(["region", "method"], observed=True)[col]
        .mean()
        .reset_index()
    )
    p = a.pivot_table(index="region", columns="method", values=col, observed=False)
    p = p.reindex(index=REGIONS, columns=METHODS).rename(
        index=REGION_LABEL, columns=METHOD_LABEL
    )
    return p.style.format(fmt).set_caption(f"{var} — {col}")


for var in VARS:
    print(
        f"\n{'=' * 70}\n{var.upper()} reliability (nominal cov68=68.3; SSR ideal 1; CE_TV ideal 0)\n{'=' * 70}"
    )
    for col, fmt in [("ssr", "{:.3f}"), ("cov68", "{:.1f}"), ("ce_tv", "{:.3f}")]:
        display(_pivot(unc_df, var, col, fmt))

# PIT rank histograms: rows = variable, cols = method, pooled over regions+seeds.
centers = (np.arange(NB_PIT) + 0.5) / NB_PIT
fig, axes = plt.subplots(
    2, len(METHODS), figsize=(3.2 * len(METHODS), 5.0), sharey=True, squeeze=False
)
for ri, var in enumerate(["t2m", "wind"]):
    for ci, method in enumerate(METHODS):
        ax = axes[ri][ci]
        h = sum(PH.get((r, var, method), np.zeros(NB_PIT)) for r in REGIONS)
        if h.sum() == 0:
            ax.axis("off")
            continue
        ax.bar(
            centers,
            h / h.sum() * NB_PIT,
            width=0.95 / NB_PIT,
            color=METHOD_COLOR[method],
            alpha=0.85,
            edgecolor="white",
            lw=0.3,
        )
        ax.axhline(1.0, color="k", lw=1, ls=":")
        ax.set_ylim(0, 2.0)
        if ri == 0:
            ax.set_title(METHOD_LABEL[method], fontsize=9)
        if ci == 0:
            ax.set_ylabel(f"{var}\nPIT density", fontsize=10, fontweight="bold")
        if ri == 1:
            ax.set_xlabel("PIT")
fig.suptitle(
    "PIT rank histograms, pooled over regions+seeds (flat = calibrated). "
    "Constant σ (left) is visibly non-uniform even where SSR≈1.",
    y=1.0,
)
fig.tight_layout()
fig.savefig(OUT_DIR / "unc_pit_histograms.png", bbox_inches="tight")
plt.show()

## 2. Resolution — does σ *discriminate* hard cases from easy ones?

This is the real question. Three complementary read-outs, all lead-independent:

- **Reliability–resolution plane** — `ρ(σ,|error|)` (resolution, →) vs `CE_TV` (miscalibration,
  ↑ better). The constant-σ baseline is pinned to `ρ=0`: **zero** resolution, whatever its
  calibration. Learned σ moves right.
- **Spread–skill relationship** — bin by σ; a resolving σ makes binned-RMSE climb the 1:1
  line, while the constant-σ model collapses to a **single point**.
- **Value of a varying σ** — `resolution gap = NLL(best constant σ) − NLL(best-scaled σ)`
  (nats): how much a *varying* σ buys once a global spread miscalibration is removed. `CV(σ)`
  reports whether σ varies at all (0 for the constant baseline). We also report the
  rank-based `ρ`, which — unlike RMSE/SSR — is insensitive to σ's overall scale and to the
  magnitude of rare large-error events, isolating whether σ *orders* cases correctly.


In [ ]:
# === Resolution: plane + spread-skill relationship + table ==================
agg = (
    unc_df.groupby(["region", "var", "method"], observed=True)
    .mean(numeric_only=True)
    .reset_index()
)

# (a) reliability–resolution plane.
fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.0))
for ax, var in zip(axes, ["t2m", "wind"], strict=False):
    sub = agg[agg["var"] == var]
    for _, r in sub.iterrows():
        ax.scatter(
            r["spearman_sig_err"],
            r["ce_tv"],
            s=70,
            color=METHOD_COLOR[r["method"]],
            edgecolor="k",
            lw=0.5,
            zorder=3,
        )
        ax.annotate(
            REGION_LABEL[r["region"]],
            (r["spearman_sig_err"], r["ce_tv"]),
            fontsize=6.5,
            xytext=(3, 3),
            textcoords="offset points",
            color="0.35",
        )
    ax.axvline(0, color="k", lw=0.8, ls=":")
    ax.set_xlabel("resolution:  Spearman ρ(σ, |error|)  →  better")
    ax.set_ylabel("miscalibration:  PIT CE$_{TV}$  (↑ = better)")
    ax.set_title(var)
    ax.set_xlim(-0.03, 0.5)
    ax.invert_yaxis()
axes[0].legend(
    handles=[
        Line2D(
            [0],
            [0],
            marker="o",
            ls="",
            mfc=METHOD_COLOR[m],
            mec="k",
            label=METHOD_LABEL[m],
        )
        for m in METHODS
    ],
    fontsize=8,
    loc="lower right",
)
fig.suptitle(
    "Does σ track true uncertainty? Reliability–resolution plane "
    "(ideal = top-right: calibrated AND resolving; constant σ pinned at ρ=0)",
    y=1.0,
)
fig.tight_layout()
fig.savefig(OUT_DIR / "unc_reliability_resolution_plane.png", bbox_inches="tight")
plt.show()


# (b) spread–skill relationship, seed-averaged curves.
def _avg_curve(cell):
    xs = [x for x, y in SS.get(cell, []) if len(x)]
    ys = [y for x, y in SS.get(cell, []) if len(y)]
    if not xs:
        return None, None
    L = min(map(len, xs))
    return np.mean([x[:L] for x in xs], 0), np.mean([y[:L] for y in ys], 0)


fig, axes = plt.subplots(
    2, len(REGIONS), figsize=(3.0 * len(REGIONS), 6.2), squeeze=False
)
for ri, var in enumerate(["t2m", "wind"]):
    for ci, region in enumerate(REGIONS):
        ax = axes[ri][ci]
        hi = 0.5
        for method in ["baseline", "tessera"]:
            bx, by = _avg_curve((region, var, method))
            if bx is None:
                continue
            ax.plot(
                bx,
                by,
                "-o",
                color=METHOD_COLOR[method],
                ms=4,
                lw=1.6,
                label=METHOD_LABEL[method],
            )
            hi = max(hi, bx.max(), by.max())
        er = agg[
            (agg["var"] == var)
            & (agg["region"] == region)
            & (agg["method"] == "era5_interp")
        ]
        if len(er):
            ax.scatter(
                er["mean_sigma"],
                er["rmse"],
                marker="*",
                s=150,
                color=METHOD_COLOR["era5_interp"],
                edgecolor="k",
                lw=0.5,
                zorder=4,
                label=METHOD_LABEL["era5_interp"],
            )
            hi = max(hi, float(er["mean_sigma"].iloc[0]), float(er["rmse"].iloc[0]))
        ax.plot([0, hi], [0, hi], "k--", lw=0.8)
        ax.set_xlim(0, hi)
        ax.set_ylim(0, hi)
        if ri == 0:
            ax.set_title(REGION_LABEL[region], fontsize=10)
        if ci == 0:
            ax.set_ylabel(f"{var}\nbinned RMSE", fontsize=10, fontweight="bold")
        if ri == 1:
            ax.set_xlabel("binned σ")
axes[0][-1].legend(fontsize=6.3, loc="upper left")
fig.suptitle(
    "Spread–skill relationship: larger predicted σ should coincide with larger error "
    "(on 1:1 = calibrated & resolving; a single ★ = no resolution)",
    y=1.0,
)
fig.tight_layout()
fig.savefig(OUT_DIR / "unc_spread_skill_relationship.png", bbox_inches="tight")
plt.show()

# (c) resolution table.
for var in VARS:
    print(
        f"\n{var.upper()} resolution  (CV σ, ρ(σ,|err|), resolution-gap nats, calib-gap nats):"
    )
    a = (
        unc_df[unc_df["var"] == var]
        .groupby(["region", "method"], observed=True)[
            ["cv_sigma", "spearman_sig_err", "resolution_gap", "calib_gap"]
        ]
        .mean()
        .reset_index()
    )
    a["region"] = a["region"].map(REGION_LABEL)
    a["method"] = a["method"].map(METHOD_LABEL)
    display(
        a.style.format(
            {
                "cv_sigma": "{:.3f}",
                "spearman_sig_err": "{:.3f}",
                "resolution_gap": "{:.3f}",
                "calib_gap": "{:.3f}",
            }
        ).hide(axis="index")
    )

## 3. Does σ track *known* drivers of uncertainty? (terrain)

The most direct test of "true uncertainty": complex terrain — large sub-grid elevation
anomaly `|Δelev|` (station elevation minus the coarse ERA5 cell) — is *independently* known to
make downscaling harder. A σ that tracks true uncertainty should be large exactly there.

We check two things per (region, variable, learned method):
1. **Per-station correlations** — does mean σ rise with `|Δelev|`, does realized RMSE rise with
   `|Δelev|`, and does per-station σ track per-station RMSE.
2. **Terrain-conditional coverage** — split observations into flat / mid / complex terrain
   terciles and compare central-68.3% coverage of **ConvCNP with vs without TESSERA** (both
   learned σ) against a **globally-calibrated constant σ** null (`σ_c = RMS(error)`, same μ).
   A σ that knows terrain holds coverage flat; a constant σ must over-cover flat terrain and
   under-cover complex terrain. (ERA5-interp has no per-station index, so the constant-σ line —
   built from each learned model's own μ — is the fair stand-in for the homoscedastic null.)


In [ ]:
# === Terrain drivers: per-station correlations + conditional coverage =======
def _pit_with_sigma(head, mu, sigma, y):
    return (_gauss if head == "gaussian" else _tnorm)(mu, sigma, y)[2]


def terrain_analysis(region, var, method, seeds=SEEDS):
    """Per-station σ vs realized RMSE vs terrain |Δelev|, and terrain-tercile
    coverage for the model's σ vs a globally-calibrated constant σ (same μ)."""
    head = VARS[var]
    se = load_station_err(region, var, method, seeds[0])
    if se is None:
        return None
    absdelev_st = np.abs(se["station_delta_elevs"].astype(np.float64))
    elev_st = se["station_elevs"].astype(np.float64)
    nst = len(elev_st)
    mus, sigs, ys, sidxs, pits, means = [], [], [], [], [], []
    for seed in seeds:
        c = load_cell(region, var, method, seed)
        if c is None or c["sidx"] is None:
            continue
        mus.append(c["mu"])
        sigs.append(c["sigma"])
        ys.append(c["y"])
        sidxs.append(c["sidx"])
        pits.append(c["pit"])
        means.append(c["mean"])
    mu = np.concatenate(mus)
    sigma = np.concatenate(sigs)
    y = np.concatenate(ys)
    sidx = np.concatenate(sidxs)
    pit = np.concatenate(pits)
    err = np.concatenate(means) - y
    cnt = np.bincount(sidx, minlength=nst).astype(np.float64)
    st_meansig = np.where(
        cnt > 0,
        np.bincount(sidx, weights=sigma, minlength=nst) / np.maximum(cnt, 1),
        np.nan,
    )
    st_rmse = np.where(
        cnt > 0,
        np.sqrt(np.bincount(sidx, weights=err**2, minlength=nst) / np.maximum(cnt, 1)),
        np.nan,
    )
    m = (
        (cnt >= 30)
        & np.isfinite(absdelev_st)
        & np.isfinite(st_meansig)
        & np.isfinite(st_rmse)
    )
    corr = dict(
        rho_sig_terrain=spearmanr(st_meansig[m], absdelev_st[m]).correlation,
        rho_rmse_terrain=spearmanr(st_rmse[m], absdelev_st[m]).correlation,
        rho_sig_rmse=spearmanr(st_meansig[m], st_rmse[m]).correlation,
    )
    absdelev_obs = absdelev_st[sidx]
    good = np.isfinite(absdelev_obs)
    q = np.quantile(absdelev_obs[good], [1 / 3, 2 / 3])
    terc = np.digitize(absdelev_obs, q)
    sigma_c = float(np.sqrt(np.mean(err**2)))  # global SSR=1 constant σ
    pit_const = _pit_with_sigma(head, mu, np.full_like(mu, sigma_c), y)
    cov = {}
    for t, name in enumerate(["flat", "mid", "complex"]):
        s = good & (terc == t)
        cov[name] = dict(
            cov_model=100 * np.mean(np.abs(pit[s] - 0.5) < C1),
            cov_const=100 * np.mean(np.abs(pit_const[s] - 0.5) < C1),
            rmse=float(np.sqrt(np.mean(err[s] ** 2))),
        )
    return dict(
        corr=corr,
        tercile=cov,
        sigma_c=sigma_c,
        st_meansig=st_meansig[m],
        st_rmse=st_rmse[m],
        st_absdelev=absdelev_st[m],
    )


TERR = {
    (region, var, method): terrain_analysis(region, var, method)
    for region in REGIONS
    for var in VARS
    for method in ["baseline", "tessera"]
}

# correlation table (tessera).
print("Per-station rank correlations (TESSERA):  σ↔terrain, RMSE↔terrain, σ↔RMSE")
rows = []
for var in VARS:
    for region in REGIONS:
        ta = TERR.get((region, var, "tessera"))
        if ta:
            rows.append(
                dict(
                    var=var,
                    region=REGION_LABEL[region],
                    **{k: round(v, 3) for k, v in ta["corr"].items()},
                )
            )
display(pd.DataFrame(rows).style.hide(axis="index"))

# terrain-conditional coverage: rows = variable, cols = region. Compares ConvCNP
# WITH vs WITHOUT TESSERA (both learned σ) against a constant-σ null that cannot
# track terrain at all.
tercs = ["flat", "mid", "complex"]
xx = np.arange(3)
fig, axes = plt.subplots(
    2, len(REGIONS), figsize=(2.9 * len(REGIONS), 6.2), sharey=True, squeeze=False
)
for ri, var in enumerate(["t2m", "wind"]):
    for ci, region in enumerate(REGIONS):
        ax = axes[ri][ci]
        tb = TERR.get((region, var, "baseline"))
        tt = TERR.get((region, var, "tessera"))
        if tt is None:
            ax.axis("off")
            continue
        if tb is not None:
            ax.plot(
                xx,
                [tb["tercile"][t]["cov_model"] for t in tercs],
                "-o",
                color=METHOD_COLOR["baseline"],
                lw=1.8,
                ms=6,
                label="ConvCNP (no TESSERA)",
            )
        ax.plot(
            xx,
            [tt["tercile"][t]["cov_model"] for t in tercs],
            "-o",
            color=METHOD_COLOR["tessera"],
            lw=1.9,
            ms=6,
            label="ConvCNP + TESSERA",
        )
        ax.plot(
            xx,
            [tt["tercile"][t]["cov_const"] for t in tercs],
            "--s",
            color="#888",
            lw=1.5,
            ms=5,
            label="constant σ (null)",
        )
        ax.axhline(68.27, color="red", lw=0.9, ls=":")
        ax.set_xticks(xx)
        ax.set_xticklabels(tercs, fontsize=9)
        ax.set_ylim(52, 90)
        if ri == 0:
            ax.set_title(REGION_LABEL[region], fontsize=10)
        if ci == 0:
            ax.set_ylabel(
                f"{var}\ncentral-68.3% cov (%)", fontsize=10, fontweight="bold"
            )
axes[0][0].legend(fontsize=6.8, loc="lower left")
fig.suptitle(
    "Conditional coverage across terrain complexity (|Δelev| terciles): ConvCNP with vs without "
    "TESSERA (both learned σ) vs a constant-σ null. Both learned σ compress the swing the "
    "constant σ shows; red dotted = nominal 68.3%.",
    y=1.0,
)
fig.tight_layout()
fig.savefig(OUT_DIR / "unc_terrain_conditional_coverage.png", bbox_inches="tight")
plt.show()

### 3b. Is elevation the right difficulty axis — or are the TESSERA latents?

Elevation `|Δelev|` (§3) is *one hand-picked* proxy for where downscaling is hard. TESSERA
latents are a learned, 16-d descriptor of the local surface. Borrowing the decomposition from
`residual_structure_analysis.ipynb`, we ask which descriptor space best **organizes** a
per-station field: take k nearest neighbours in a space (geographic / terrain `[elev, Δelev]` /
z-scored lat16), measure how smooth the field is across those neighbours, and normalise against
a random-neighbour reference into a

`structure score = 1 − roughness / roughness_random`   (0 = no structure, →1 = coherent).

- **Difficulty field = per-station realized RMSE** — the model-agnostic ground truth of *where
  it is hard*. If `latent > terrain`, the learned embedding locates difficulty better than
  elevation. **This is the non-circular test** (no σ involved).
- We also score the **σ field**, to see whether σ is organised the same way. Caveat: TESSERA's σ
  is *built from* the latents, so "TESSERA-σ is latent-organised" is partly tautological — the
  decisive comparisons are the realized-error field and the *baseline's* σ.

Restricted to regions with ≥ 30 well-sampled (`count ≥ 30`) stations; **Australia is dropped**
(only 7 stations clear that bar). Small-n regions (East Asia, S. Africa) are noisier — read
Europe and the US as the load-bearing evidence.


In [ ]:
# === 3b. Which descriptor space organizes the difficulty / σ field? =========
# Reuses the residual-structure recipe: structure score = 1 − rough/rough_random,
# with neighbours in geographic / terrain[elev,Δelev] / z-scored TESSERA-lat16 space.
from scipy.spatial import cKDTree

SR_KNN = 8
MIN_N = 30

_LAT_CACHE = {}


def _latents_lut(cfg_path):
    cfg = json.load(open(cfg_path))
    lp, cp = cfg.get("vae_latents_path"), cfg.get("vae_latents_station_csv")
    if not lp or not cp or not Path(lp).exists() or not Path(cp).exists():
        return None
    if (lp, cp) not in _LAT_CACHE:
        arr = np.load(lp)
        ids = pd.read_csv(cp)["station_id"].astype(str).values
        _LAT_CACHE[(lp, cp)] = {str(s): arr[i] for i, s in enumerate(ids)}
    return _LAT_CACHE[(lp, cp)]


def _zscore(A):
    A = np.asarray(A, float)
    return (A - A.mean(0)) / (A.std(0) + 1e-8)


def _knn(coords, k):
    _, idx = cKDTree(coords).query(coords, k=k + 1)
    return idx[:, 1:]


def _rough(v, nn):
    return float(np.mean(np.abs(v - v[nn].mean(1))))


def _rough_rand(v, k, seed=0):
    n = len(v)
    k = min(k, n - 1)
    nd = int(np.clip(30000 // max(n, 1), 60, 500))
    rng = np.random.default_rng(seed)
    acc = 0.0
    for _ in range(nd):
        R = rng.random((n, n))
        np.fill_diagonal(R, np.inf)
        acc += _rough(v, np.argpartition(R, k, axis=1)[:, :k])
    return acc / nd


def _station_sigma(region, var, method, nst):
    ssum = np.zeros(nst)
    cnt = np.zeros(nst)
    for seed in SEEDS:
        c = load_cell(region, var, method, seed)
        if c is None or c["sidx"] is None:
            continue
        ssum += np.bincount(c["sidx"], weights=c["std"], minlength=nst)[:nst]
        cnt += np.bincount(c["sidx"], minlength=nst)[:nst]
    return np.where(cnt > 0, ssum / np.maximum(cnt, 1), np.nan)


sr_rows = []
for region in REGIONS:
    for var in VARS:
        lut = _latents_lut(
            root(region) / f"{stem(var, 'tessera')}_seed{SEEDS[0]}" / "config.json"
        )
        if lut is None:
            continue
        se = load_station_err(region, var, "tessera", SEEDS[0])
        if se is None:
            continue
        sid = se["station_ids"].astype(str)
        nst = len(sid)
        keep = (
            (se[f"{var}_station_count"].astype(float) >= MIN_N)
            & np.isfinite(se["station_delta_elevs"].astype(float))
            & np.array([s in lut for s in sid])
        )
        idx = np.where(keep)[0]
        if len(idx) < SR_KNN + 2 or len(idx) < 12:
            print(f"[skip] {region} {var}: only {len(idx)} stations ≥{MIN_N} obs")
            continue
        lat = se["station_lats"].astype(float)[idx]
        lon = se["station_lons"].astype(float)[idx]
        Lz = _zscore(np.array([lut[s] for s in sid[idx]]))
        spaces = {
            "geo": _knn(
                np.column_stack([lon * np.cos(np.radians(lat.mean())), lat]), SR_KNN
            ),
            "terrain": _knn(
                _zscore(
                    np.column_stack(
                        [
                            se["station_elevs"].astype(float)[idx],
                            se["station_delta_elevs"].astype(float)[idx],
                        ]
                    )
                ),
                SR_KNN,
            ),
            "latent": _knn(Lz, SR_KNN),
        }
        fields = {"RMSE(tessera)": se[f"{var}_station_rmse"].astype(float)[idx]}
        for method in ["baseline", "tessera"]:
            fields[f"σ({method})"] = _station_sigma(region, var, method, nst)[idx]
        for fname, fv in fields.items():
            if not np.all(np.isfinite(fv)):
                continue
            rr = _rough_rand(fv, SR_KNN)
            sr_rows.append(
                dict(
                    region=region,
                    var=var,
                    field=fname,
                    n=len(idx),
                    **{sp: 1 - _rough(fv, nn) / rr for sp, nn in spaces.items()},
                )
            )
sr_df = pd.DataFrame(sr_rows)

# table for the difficulty field (the non-circular answer to "elevation vs latent").
print(
    "\nStructure score of the DIFFICULTY field (per-station realized RMSE) — "
    "latent > terrain ⇒ latents locate difficulty better than elevation:"
)
diff = sr_df[sr_df.field == "RMSE(tessera)"].copy()
diff["region"] = diff["region"].map(REGION_LABEL)
diff["latent>terrain"] = diff["latent"] > diff["terrain"]
display(
    diff[["region", "var", "n", "geo", "terrain", "latent", "latent>terrain"]]
    .style.format({"geo": "{:.3f}", "terrain": "{:.3f}", "latent": "{:.3f}"})
    .hide(axis="index")
)

# figure: difficulty-field structure score, terrain vs latent, per region × variable.
order = [r for r in REGIONS if (sr_df.region == r).any()]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), squeeze=False)
w = 0.38
for ci, var in enumerate(["t2m", "wind"]):
    ax = axes[0][ci]
    sub = (
        sr_df[(sr_df["var"] == var) & (sr_df.field == "RMSE(tessera)")]
        .set_index("region")
        .reindex(order)
    )
    x = np.arange(len(order))
    ax.bar(x - w / 2, sub["terrain"], w, color="#8c6d31", label="terrain (elev, Δelev)")
    ax.bar(x + w / 2, sub["latent"], w, color="#2ca02c", label="TESSERA lat16")
    ax.axhline(0, color="k", lw=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels([REGION_LABEL[r] for r in order], rotation=20, ha="right")
    ax.set_title(f"{var} — difficulty (realized RMSE) field")
    ax.set_ylabel("structure score")
    for xi, r in zip(x, order, strict=False):
        ax.annotate(
            f"n={int(sub.loc[r, 'n'])}",
            (xi, 0.002),
            fontsize=6.5,
            ha="center",
            va="bottom",
            color="0.4",
        )
    ax.legend(fontsize=8, loc="upper right")
fig.suptitle(
    "Which descriptor locates where downscaling is hard? Higher = better. "
    "Latents ≫ terrain for wind; terrain is competitive for t2m (elevation-dominated).",
    y=1.02,
)
fig.tight_layout()
fig.savefig(OUT_DIR / "unc_difficulty_descriptor.png", bbox_inches="tight")
plt.show()

## 4. Per-region × per-variable synthesis

One master scorecard: point skill (RMSE), reliability (SSR, coverage, CE_TV), and resolution
(CV σ, ρ(σ,|err|), resolution-gap) for every (region, variable, method), seed mean ± std,
exported to CSV. The reliability–resolution plane above is its visual summary.


In [ ]:
# === Master scorecard + CSV export ==========================================
METRICS = [
    "rmse",
    "ssr",
    "cov68",
    "ce_tv",
    "cv_sigma",
    "spearman_sig_err",
    "resolution_gap",
    "calib_gap",
    "nll_model",
]
scorecard = unc_df.groupby(["var", "region", "method"], observed=True)[METRICS].agg(
    ["mean", "std"]
)
scorecard.columns = [f"{m}_{s}" for m, s in scorecard.columns]
scorecard = scorecard.reset_index()
scorecard["region"] = scorecard["region"].map(REGION_LABEL)
scorecard["method"] = scorecard["method"].map(METHOD_LABEL)
scorecard.to_csv(OUT_DIR / "uncertainty_scorecard.csv", index=False)


def _cell(m, s):
    return (
        ""
        if pd.isna(m)
        else (f"{m:.3f}±{s:.3f}" if pd.notna(s) and s > 0 else f"{m:.3f}")
    )


for var in VARS:
    print(
        f"\n{'=' * 80}\n{var.upper()} — uncertainty scorecard (seed mean±std)\n{'=' * 80}"
    )
    sub = scorecard[scorecard["var"] == var]
    show = pd.DataFrame({"region": sub["region"], "method": sub["method"]})
    for col, lab in [
        ("rmse", "RMSE"),
        ("ssr", "SSR"),
        ("cov68", "cov68%"),
        ("ce_tv", "CE_TV"),
        ("cv_sigma", "CV(σ)"),
        ("spearman_sig_err", "ρ(σ,|e|)"),
        ("resolution_gap", "resGap"),
        ("nll_model", "NLL"),
    ]:
        show[lab] = [
            _cell(m, s)
            for m, s in zip(sub[f"{col}_mean"], sub[f"{col}_std"], strict=False)
        ]
    display(show.style.hide(axis="index"))

print("\nWrote", OUT_DIR / "uncertainty_scorecard.csv")
print("Figures:", sorted(p.name for p in OUT_DIR.glob("unc_*.png")))

## 5. Verdict

**How do we know the "true uncertainty"?** We never see it per case — only one realization.
We recover it by **conditioning and pooling**: the realized error within a set of cases that
share a predicted σ (or a covariate like terrain) *is* the true average uncertainty for that
set. A good σ must satisfy `E[(y−μ)²|σ]=σ²`, and must do so **conditionally**, not just on
average.

**Does σ track the true uncertainty?**
- **Reliability alone is a trap.** The constant-σ ERA5 reference reaches `SSR≈0.93–1.0` yet has
  `CV(σ)=0`, `ρ(σ,|error|)=0`, and a resolution gap of **0 nats** in every region and variable —
  its PIT is the most ∩-shaped of the three. "σ ≈ RMSE on average" is necessary but carries
  *no* case-level information.
- **The learned ConvCNP σ genuinely resolves.** Its σ varies (`CV(σ)≈0.25–0.55`), ranks errors
  (`ρ(σ,|error|)≈0.12–0.42`, largest for European t2m), climbs the spread–skill 1:1 line, and
  buys up to `~0.18` nats over the best constant σ. Per-station, σ rises with terrain
  complexity (`ρ(σ,|Δelev|)≈0.3–0.57`) and tracks realized RMSE (`ρ≈0.7`), and it **compresses
  the terrain-driven coverage swing** — a globally-calibrated constant σ over-covers flat and
  under-covers complex terrain (~10-point swing), the learned σ holds it to ~2–6 points.
  Cleanest for t2m; genuine but weaker for wind, and it does **not** fully remove residual
  under-coverage in the most complex terrain.
- **Elevation is not always the right difficulty axis (§3b).** Scoring which descriptor space
  *organises the realized error*: for **t2m**, terrain (elev, Δelev) is as good as the TESSERA
  latents (temperature error is elevation-dominated), so elevation is a justified proxy; for
  **wind**, the latents locate difficulty far better than elevation (Europe structure score
  0.34 vs 0.22; US 0.22 vs 0.09) — wind's sub-grid hardness is roughness / land-cover /
  exposure, which elevation misses but the embedding encodes. The *right* axis to test σ
  against is variable-dependent, and for wind it is the learned latent, not elevation.

**Does TESSERA make σ track uncertainty better?**
- **t2m: yes, everywhere** — higher resolution (`ρ` and resolution gap up in every region) and
  better or equal calibration (lower `CE_TV`, coverage nearer nominal). Largest gains in
  data-rich, terrain-rich Europe / East Asia.
- **wind: improved but region-dependent.** TESSERA lowers RMSE in every region, the US
  included (1.85 vs 1.96), and its resolution is comparable-to-slightly-better on average
  (mean `ρ` 0.21 vs 0.20; resolution gap 0.065 vs 0.053 nats). But the gains are uneven:
  resolution is strong in East Asia (`ρ≈0.35`) and weak in the US (`ρ≈0.08–0.12` for *both*
  variants). The US is the hardest wind case — both learned models are **over-confident** there
  (central-68% coverage ≈55%, `SSR≈0.87`), so σ under-covers regardless of TESSERA.

**Bottom line.** A learned, heteroscedastic σ does track the true uncertainty — substantially
so for temperature, more partially for wind — whereas a constant σ, however well tuned on
average, cannot. TESSERA sharpens that tracking for t2m across all regions; for wind the
picture is regional and tail-sensitive. Uncertainty quality is therefore best read on **two
axes at once** (reliability *and* resolution), per region and per variable — the reliability–
resolution plane is the one-glance summary.

---
*Caveats.* (i) The ERA5-interp PIT uses a Gaussian CDF even for wind (its σ is a marginal
constant), so its wind coverage is only indicative — the metrics that matter for it (`CV=0`,
`ρ=0`, resolution gap `=0`) are exact regardless. (ii) The constant-σ terrain counterfactual
is built from each learned model's own μ, isolating the effect of *varying* σ. (iii) Wind CRPS
elsewhere uses an ensemble estimator; here we report NLL (closed form, exact for both heads).
(iv) `resolution_gap` and `calib_gap` hold μ fixed, so they score the σ head, not the point
predictor. (v) The truncated-Normal mean / std / PIT use numerically stable log-CDF (`log_ndtr`)
forms matching `evaluate.py`; the naive `1−Φ(−μ/σ)` closed form underflows for calm wind
(μ/σ ≪ 0) and would spuriously inflate wind RMSE — every cell's RMSE here matches the stored
`rmse_at_mean` in `test_summary.json`.
